# 上下文工程

[上下文工程](https://docs.langchain.com/oss/python/langchain/context-engineering)（Context Engineering）对于 Agent 得出正确的结果至关重要。模型回答不好，很多时候不是因为能力不足，而是因为没有获得足以推断出正确结果的上下文信息。通过上下文工程，增强 Agent 获取和管理上下文的能力，是很有必要的。

**LangGraph 将上下文分为三种类型：**

- 模型上下文（Model Context）
- 工具上下文（Tool Context）
- 生命周期上下文（Life-cycle Context）

无论哪种 Context，都需要定义它的 Schema。在这方面，LangGraph 提供了相当高的自由度，你可以使用 `dataclasses`、`pydantic`、`TypedDict` 这些包的任意一个创建你的 Context Schema。

In [1]:
# uv add ipynbname

In [2]:
import os
import uuid
import sqlite3

from typing import Callable
from dotenv import load_dotenv
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.middleware import (
    dynamic_prompt,
    wrap_model_call,
    ModelRequest,
    ModelResponse,
    SummarizationMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.sqlite import SqliteStore

# 加载模型配置
_ = load_dotenv()

# 加载模型
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3.8-flash",
    temperature=0.7,
)

## 一、动态修改系统提示词

上下文工程与前序章节的中间件（middleware）和记忆（memory）密不可分。上下文的具体实现依赖中间件，而上下文的存储则依赖记忆系统。具体来讲，LangGraph 预置了 `@dynamic_prompt` 中间件，用于动态修改系统提示词。

既然是动态修改，肯定需要某个条件来触发修改。除了开发触发逻辑，我们还需要从智能体中获取触发逻辑所需的即时变量。这些变量通常存储在以下三个存储介质中：

- 运行时（Runtime）- 所有节点共享一个 Runtime。同一时刻，所有节点取到的 Runtime 的值是相同的。一般用于存储时效性要求较高的信息。
- 短期记忆（State）- 在节点之间按顺序传递，每个节点接收上一个节点处理后的 State。主要用于存储 Prompt 和 AI Message。
- 长期记忆（Store）- 负责持久化存储，可以跨 Workflow / Agent 保存信息。可以用来存用户偏好、以前算过的统计值等。

以下三个例子，分别演示如何使用来自 Runtime、State、Store 中的上下文，编写触发条件。

### 1）使用 `State` 管理上下文

利用 `State` 中蕴含的信息操纵 system prompt。

In [3]:
@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    base = "You are a helpful assistant."

    if message_count > 6:
        base += "\nThis is a long conversation - be extra concise."

    # 临时打印base看效果
    print(base)

    return base


agent = create_agent(model=llm, middleware=[state_aware_prompt])

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "广州今天的天气怎么样？"},
            {"role": "assistant", "content": "广州天气很好"},
            {"role": "user", "content": "吃点什么好呢"},
            {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
            {"role": "user", "content": "香茅是什么"},
            {
                "role": "assistant",
                "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉",
            },
            {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
        ]
    },
)

for message in result["messages"]:
    message.pretty_print()

You are a helpful assistant.
This is a long conversation - be extra concise.


================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================

广州天气很好
================================ Human Message =================================

吃点什么好呢
================================== Ai Message ==================================

要不要吃香茅鳗鱼煲
================================ Human Message =================================

香茅是什么
================================== Ai Message ==================================

香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== Ai Message ==================================

走起！来一锅香茅鳗鱼煲，再配点米饭和青菜，开吃


把 `message_count > 6` 里的 6 改成 7，试试看会发生什么。

### 2）使用 `Store` 管理上下文

In [4]:
@dataclass
class Context:
    user_id: str


@dynamic_prompt
def store_aware_prompt(request: ModelRequest) -> str:
    user_id = request.runtime.context.user_id

    # Read from Store: get user preferences
    store = request.runtime.store
    user_prefs = store.get(("preferences",), user_id)

    base = "You are a helpful assistant."

    if user_prefs:
        style = user_prefs.value.get("communication_style", "balanced")
        base += f"\nUser prefers {style} responses."

    return base


store = InMemoryStore()

agent = create_agent(
    model=llm,
    middleware=[store_aware_prompt],
    context_schema=Context,
    store=store,
)

# 预置两条偏好信息
store.put(("preferences",), "user_1", {"communication_style": "Chinese"})
store.put(("preferences",), "user_2", {"communication_style": "Korean"})

In [5]:
# 用户1喜欢中文回复
result = agent.invoke(
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful assistant. Please be extra concise.",
            },
            {"role": "user", "content": 'What is a "hold short line"?'},
        ]
    },
    context=Context(user_id="user_1"),
)

for message in result["messages"]:
    message.pretty_print()

================================ System Message ================================

You are a helpful assistant. Please be extra concise.
================================ Human Message =================================

What is a "hold short line"?
================================== Ai Message ==================================

“Hold short line”（等待线/跑道等待线）是机场跑道入口处的标线，要求飞机或车辆停在线外，得到空管许可后才能进入跑道。


In [6]:
# 用户2喜欢韩文回复
result = agent.invoke(
    {
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful assistant. Please be extra concise.",
            },
            {"role": "user", "content": 'What is a "hold short line"?'},
        ]
    },
    context=Context(user_id="user_2"),
)

for message in result["messages"]:
    message.pretty_print()

================================ System Message ================================

You are a helpful assistant. Please be extra concise.
================================ Human Message =================================

What is a "hold short line"?
================================== Ai Message ==================================

항공기의 활주로 또는 보호구역 진입을 방지하기 위해 반드시 정지해야 하는 유도로(Taxiway)의 노란색 표지선입니다.


### 3）使用 `Runtime` 管理上下文

In [7]:
@dataclass
class Context:
    user_role: str
    deployment_env: str


@dynamic_prompt
def context_aware_prompt(request: ModelRequest) -> str:
    # Read from Runtime Context: user role and environment
    user_role = request.runtime.context.user_role
    env = request.runtime.context.deployment_env

    base = "You are a helpful assistant."

    if user_role == "admin":
        base += "\nYou can use the get_weather tool."
    else:
        base += "\nYou are prohibited from using the get_weather tool."

    if env == "production":
        base += "\nBe extra careful with any data modifications."

    return base


@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=llm,
    tools=[get_weather],
    middleware=[context_aware_prompt],
    context_schema=Context,
    checkpointer=InMemorySaver(),
)

In [8]:
# 利用 Runtime 中的两个变量，动态控制 System prompt
# 将 user_role 设为 admin，允许使用天气查询工具
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "广州今天的天气怎么样？"}]},
    context=Context(user_role="admin", deployment_env="production"),
    config=config,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_b59c6e9077f846ff9be08463)
 Call ID: call_b59c6e9077f846ff9be08463
  Args:
    city: 广州
================================= Tool Message =================================
Name: get_weather

It's always sunny in 广州!
================================== Ai Message ==================================

根据查询结果，广州目前是**晴天** ☀️。

不过需要提醒一下：这次返回的数据看起来像是一个固定的默认值（提示内容为"广州永远是晴天"），并不是实时的气象观测数据，因此准确性无法保证。

建议你通过以下方式确认今天的实际天气：
- 查看手机自带的天气 App
- 访问中国天气网（weather.com.cn）或广州市气象局官网
- 搜索"广州天气"获取实时温度、降雨概率和预警信息

如果你还需要我查询其他城市，或者有其他问题，随时告诉我。


In [9]:
# 若将 user_role 改为 viewer，则无法使用天气查询工具
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "广州今天的天气怎么样？"}]},
    context=Context(user_role="viewer", deployment_env="production"),
    config=config,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================

抱歉，我目前无法为你查询实时天气数据（我的天气查询功能当前不可用）。

建议你通过以下方式获取广州今天的准确天气：
- 中国天气网（weather.com.cn）
- 手机自带的天气应用
- 搜索引擎直接搜索"广州天气"

如果你有其他我可以帮忙的问题，欢迎随时告诉我！


In [10]:
result["messages"]

[HumanMessage(content='广州今天的天气怎么样？', additional_kwargs={}, response_metadata={}, id='e9185342-e655-4d1f-8079-f2239ccbbbfe'),
 AIMessage(content='抱歉，我目前无法为你查询实时天气数据（我的天气查询功能当前不可用）。\n\n建议你通过以下方式获取广州今天的准确天气：\n- 中国天气网（weather.com.cn）\n- 手机自带的天气应用\n- 搜索引擎直接搜索"广州天气"\n\n如果你有其他我可以帮忙的问题，欢迎随时告诉我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 338, 'total_tokens': 482, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 73, 'rejected_prediction_tokens': None, 'text_tokens': 144}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': 338}}, 'model_provider': 'openai', 'model_name': 'qwen3.8-flash', 'system_fingerprint': None, 'id': 'chatcmpl-bae9bb88-9daf-44d8-9c5e-5c7b9e8da7f2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f697-e16a-77d4-8ab3-4d6d6b565594-0', tool_calls=[], invali

## 二、动态修改消息列表

LangGraph 预制了动态修改消息列表（Messages）的中间件 `@wrap_model_call`。上一节已经演示如何从 `State`、`Store`、`Runtime` 中获取上下文，本节将不再一一演示。在下面这个例子中，我们主要演示如何使用 `Runtime` 将本地文件的内容注入消息列表。

In [11]:
@dataclass
class FileContext:
    uploaded_files: list[dict]


@wrap_model_call
def inject_file_context(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Inject context about files user has uploaded this session."""
    uploaded_files = request.runtime.context.uploaded_files

    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except Exception:
        try:
            import ipynbname

            notebook_path = ipynbname.path()
            base_dir = os.path.dirname(notebook_path)
        except Exception:
            # 交互式 Lab 4 与 headless 环境下 ipynbname 均无法反查路径，回退到当前目录
            base_dir = os.getcwd()

    file_sections = []
    for file in uploaded_files:
        name, ftype = "", ""
        path = file.get("path")
        if path:
            base_filename = os.path.basename(path)
            stem, ext = os.path.splitext(base_filename)
            name = stem or base_filename
            ftype = ext.lstrip(".") if ext else None

            # 构建文件描述内容
            content_list = [f"名称: {name}"]
            if ftype:
                content_list.append(f"类型: {ftype}")

            # 解析相对路径为绝对路径
            abs_path = path if os.path.isabs(path) else os.path.join(base_dir, path)

            # 读取文件内容
            content_block = ""
            if abs_path and os.path.exists(abs_path):
                try:
                    with open(abs_path, "r", encoding="utf-8") as f:
                        content_block = f.read()
                except Exception as e:
                    content_block = f"[读取文件错误 '{abs_path}': {e}]"
            else:
                content_block = "[文件路径缺失或未找到]"

            section = f"---\n{chr(10).join(content_list)}\n\n{content_block}\n---"
            file_sections.append(section)

    file_context = f"已加载的会话文件：\n{chr(10).join(file_sections)}\n回答问题时请参考这些文件。"

    # Inject file context before recent messages
    messages = [
        *request.messages,
        {"role": "user", "content": file_context},
    ]
    request = request.override(messages=messages)

    return handler(request)


agent = create_agent(
    model=llm,
    middleware=[inject_file_context],
    context_schema=FileContext,
)

In [12]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "关于上海地铁的无脸乘客，有什么需要注意的？",
            }
        ],
    },
    context=FileContext(uploaded_files=[{"path": "./docs/rule_horror.md"}]),
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

关于上海地铁的无脸乘客，有什么需要注意的？
================================== Ai Message ==================================

根据已加载的《上海规则怪谈》，关于**上海地铁上的无脸乘客**，需要注意以下几点：

1. **触发条件**  
   地铁运营结束后，如果你仍然身处车厢内，就可能会出现一位**无脸乘客**。

2. **它会问你什么**  
   这位无脸乘客会低声问你：  
   **“你要去哪？”**

3. **必须怎么回答**  
   你只能报一个**真实存在的上海地名**。  
   例如可以说真实存在的地铁站名、路名、区名、地标名等，但必须是现实中确实存在的上海地名。

4. **绝对不能做的事**  
   - 不能说**不存在的地名**。  
   - 不能保持沉默。  
   - 不能回答模糊、编造、玩笑式或虚构的地点。

5. **违反后果**  
   如果说出不存在的地名，或者保持沉默，你会在车厢内看到**自己的尸体**。

简单总结：  
**地铁停运后若遇到无脸乘客问“你要去哪”，必须立刻回答一个真实存在的上海地名，不能说假地名，也不能不说话。**


## 三、在工具中使用上下文

下面，我们尝试在工具中使用存储在 `SqliteStore` 中的上下文信息。

In [13]:
# 删除SQLite数据库
if os.path.exists("user-info.db"):
    os.remove("user-info.db")

# 创建SQLite存储
conn = sqlite3.connect("user-info.db", check_same_thread=False, isolation_level=None)
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA busy_timeout = 30000;")

store = SqliteStore(conn)

# 预置两条用户信息
store.put(
    ("user_info",),
    "柳如烟",
    {
        "description": "清冷才女，身怀绝技，为寻身世之谜踏入江湖。",
        "birthplace": "吴兴县",
    },
)
store.put(
    ("user_info",),
    "苏慕白",
    {
        "description": "孤傲剑客，剑法超群，背负家族血仇，隐于市井追寻真相。",
        "birthplace": "杭县",
    },
)

### 1）基础用例

使用 `ToolRuntime`

In [14]:
@tool
def fetch_user_data(user_id: str, runtime: ToolRuntime) -> str:
    """
    Fetch user information from the in-memory store.

    :param user_id: The unique identifier of the user.
    :param runtime: The tool runtime context injected by the framework.
    :return: The user's description string if found; an empty string otherwise.
    """
    store = runtime.store
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        user_desc = user_info.value.get("description", "")

    return user_desc


agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,
)

In [15]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "五分钟之内，我要柳如烟的全部信息"}]}
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

五分钟之内，我要柳如烟的全部信息
================================== Ai Message ==================================

抱歉，我目前只有一个通过 `user_id` 查询用户信息的工具，而您提供的"柳如烟"是一个人名，我无法直接将其作为 `user_id` 进行查询。

不过，让我先尝试一下：
Tool Calls:
  fetch_user_data (call_72b58b44ff8c4ce7b6f910ff)
 Call ID: call_72b58b44ff8c4ce7b6f910ff
  Args:
    user_id: 柳如烟
================================= Tool Message =================================
Name: fetch_user_data

清冷才女，身怀绝技，为寻身世之谜踏入江湖。
================================== Ai Message ==================================

查到了，不过先说清楚：我这边能用的查询工具只有一个，即按 `user_id` 拉取用户描述，没有更详细的信息库（比如背景设定、关系网、剧情线之类）。所以"全部信息"实际上就是下面这一条：

**柳如烟**
> 清冷才女，身怀绝技，为寻身世之谜踏入江湖。

**信息要点拆解：**
- 性格/气质：清冷
- 定位：才女（文才出众）
- 能力：身怀绝技（武艺或某种专长未具体说明）
- 动机：寻找身世之谜
- 行动：踏入江湖

如果你手上还有她对应的 `user_id`（可能是编号、拼音或其他代号），发给我，我再查一次，看能不能拿到更完整的内容。


### 2）复杂一点的例子

使用 `ToolRuntime[Context]`

In [16]:
@dataclass
class Context:
    key: str


@tool
def fetch_user_data(user_id: str, runtime: ToolRuntime[Context]) -> str:
    """
    Fetch user information from the in-memory store.

    :param user_id: The unique identifier of the user.
    :param runtime: The tool runtime context injected by the framework.
    :return: The user's description string if found; an empty string otherwise.
    """
    key = runtime.context.key

    store = runtime.store
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        user_desc = user_info.value.get(key, "")

    return f"{key}: {user_desc}"


agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,
)

In [17]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "五分钟之内，我要柳如烟的全部信息"}]},
    context=Context(key="birthplace"),
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

五分钟之内，我要柳如烟的全部信息
================================== Ai Message ==================================
Tool Calls:
  fetch_user_data (call_57326f57a1634b74966b5412)
 Call ID: call_57326f57a1634b74966b5412
  Args:
    user_id: 柳如烟
================================= Tool Message =================================
Name: fetch_user_data

birthplace: 吴兴县
================================== Ai Message ==================================

**柳如烟 — 现有档案记录**

| 字段 | 内容 |
|------|------|
| 出生地 | 吴兴县 |

目前检索到的信息仅此一条：出生地为**吴兴县**。

若你需要更完整的资料（履历、社会关系、行踪、其他身份信息等），请补充以下任一线索，我可继续定向追查：

1. **身份背景** — 她是做什么的？（商户、艺人、官员家眷、江湖人物等）
2. **时间地点** — 你关注的是哪个年代、哪个地区的柳如烟？
3. **关联人或事** — 有无相关的人物、机构、事件名可作为交叉检索的锚点？
4. **数据源** — 是否有指定的档案库、名录或文档需要我去查阅？

给我方向，剩下的交给我。


### 四、压缩上下文

LangChain 提供了内置的中间件 `SummarizationMiddleware` 用于压缩上下文。该中间件维护的是典型的 **生命周期上下文**，与 **模型上下文** 和 **工具上下文** 的瞬态更新不同，生命周期上下文会持续更新：持续将旧消息替换为摘要。

除非上下文超长，导致模型能力降低，否则不需要使用 `SummarizationMiddleware`。一般来说，触发摘要的值可以设得较大。比如：

- `max_tokens_before_summary`: 3000
- `messages_to_keep`: 20

> 如果你想了解更多关于上下文腐坏（Context Rot）的信息，Chroma 团队在 2025 年 7 月 14 日发布的 [*Context Rot: How Increasing Input Tokens Impacts LLM Performance*](https://research.trychroma.com/context-rot)，系统性地揭示了长上下文导致模型性能退化的现象。

In [18]:
# 创建短期记忆
checkpointer = InMemorySaver()

# 创建带内置摘要中间件的Agent
# 为了让配置能在我们的例子里生效，这里的触发值设得很小
agent = create_agent(
    model=llm,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 40),  # Trigger summarization at 40 tokens
            keep=("messages", 1),  # Keep last 1 messages after summary
        ),
    ],
)

In [19]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "广州今天的天气怎么样？"},
            {"role": "assistant", "content": "广州天气很好"},
            {"role": "user", "content": "吃点什么好呢"},
            {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
            {"role": "user", "content": "香茅是什么"},
            {
                "role": "assistant",
                "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉",
            },
            {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
        ]
    },
    checkpointer=checkpointer,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is asking casual lifestyle questions: current Guangzhou weather, food suggestions, and clarification about an ingredient.

## SUMMARY
- The user asked about Guangzhou’s weather today.
- The assistant replied that Guangzhou’s weather is very good.
- The user asked what to eat.
- The assistant suggested “香茅鳗鱼煲” (lemongrass eel pot).
- The user asked what “香茅” is.
- The assistant explained that 香茅 is also called 柠檬草 (lemongrass) and is commonly used in Thai tom yum soup and Vietnamese grilled meat dishes.

## ARTIFACTS
None.

## NEXT STEPS
No explicit tasks remain. The assistant can wait for the user’s next question or provide further details about lemongrass, eel pot, or Guangzhou weather/food if asked.
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== 